# CNIBP One-Click (VSCode + Colab)\n在 VSCode 的 Colab 插件连接远程 runtime 后，从上到下 `Run All`。

In [ ]:
# 只需要改这3项
GIT_REPO = 'https://github.com/67vmg9wrfn-beep/Lab.git'
GIT_BRANCH = 'main'
PROJECT_SUBDIR = '06_experiments/cnibp/repro_ppg_bp'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
if os.path.exists('/content/repo_src'):
    shutil.rmtree('/content/repo_src')
subprocess.run(['git','clone','--depth','1','--branch', GIT_BRANCH, GIT_REPO, '/content/repo_src'], check=True)
print('git clone done')


In [ ]:
import os
PROJECT_ROOT = f'/content/repo_src/{PROJECT_SUBDIR}'
print('PROJECT_ROOT=', PROJECT_ROOT)
assert os.path.exists(PROJECT_ROOT), f'Path not found: {PROJECT_ROOT}'

In [ ]:
from pathlib import Path
import re

BASE = Path('/content/drive/MyDrive')
if not BASE.exists():
    raise FileNotFoundError('Drive not mounted at /content/drive/MyDrive')

# Find all .mat files recursively under MyDrive
mat_files = list(BASE.rglob('*.mat'))
print(f'[INFO] total .mat files found: {len(mat_files)}')

# Match filenames like Part_0.mat ... Part_4.mat (case-insensitive)
pat = re.compile(r'^part_([0-4])\.mat$', re.IGNORECASE)
by_dir = {}
for f in mat_files:
    m = pat.match(f.name)
    if not m:
        continue
    d = str(f.parent)
    by_dir.setdefault(d, set()).add(int(m.group(1)))

candidates = [d for d, idxs in by_dir.items() if idxs == {0,1,2,3,4}]

if not candidates:
    print('[DEBUG] candidate dirs and matched parts:')
    for d, idxs in sorted(by_dir.items()):
        print(' ', d, '=>', sorted(list(idxs)))
    # Extra hint: show first 30 mat file paths
    print('[DEBUG] first 30 .mat paths:')
    for f in mat_files[:30]:
        print(' ', f)
    raise FileNotFoundError('Could not locate a directory containing Part_0.mat..Part_4.mat (case-insensitive).')

# Prefer path containing kachuee/raw_mat if multiple
preferred = [d for d in candidates if 'kachuee' in d.lower() and 'raw_mat' in d.lower()]
DATA_ROOT = preferred[0] if preferred else candidates[0]
print('[OK] DATA_ROOT =', DATA_ROOT)
if len(candidates) > 1:
    print('[INFO] multiple candidates found:')
    for d in candidates:
        print(' ', d)


In [ ]:
import subprocess
subprocess.run(['python','-m','pip','install','-r', f'{PROJECT_ROOT}/requirements_colab.txt'], check=True)
subprocess.run(['python','-m','pip','install','-e', PROJECT_ROOT], check=True)
print('dependencies installed')


In [ ]:
import os, subprocess
OUT_ROOT='/content/drive/MyDrive/cnibp_repro_outputs'
os.makedirs(OUT_ROOT, exist_ok=True)
log_file=f'{OUT_ROOT}/last_run.log'
cmd=[
    'python','-m','cnibp_repro.run_repro',
    '--drive_root', DATA_ROOT,
    '--config',f'{PROJECT_ROOT}/configs/paper_repro.json',
    '--output_root',OUT_ROOT
]
with open(log_file, 'w', encoding='utf-8') as f:
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        f.write(line)
    code=proc.wait()
if code != 0:
    raise RuntimeError(f'run_repro failed, see {log_file}')
print(f'log saved: {log_file}')


In [ ]:
# 失败时你只需要看这个输出路径
print('/content/drive/MyDrive/cnibp_repro_outputs/last_run.log')